# Project - AirLine AI Assistant

In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json

In [5]:
# Intialization

load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"OpenAI API key is present and it begins with {api_key[:8]}")
else:
    print("Key Not Found - OpenAI API Key")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API key is present and it begins with sk-proj-


In [6]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [12]:
def chat(message, history):
    history = [{"role":h["role"], "content" : h["content"]} for h in history]
    messages = [{"role" : "system", "content" : system_message}] + history + [{"role" : "user", "content" : message}]
    response = openai.chat.completions.create(model = MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn = chat).launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.


In [15]:
ticket_prices = {"london" : "$799", "paris" : "$899", "tokyo" : "$1400", "berlin":"$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown Ticket Price")
    return f"The price of a ticket to {destination_city} is {price}"

In [16]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [71]:
price_function = {
    "name" : "get_ticket_price",
    "description" : "Get the price of a return ticket to the destination city.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city": {
                "type" : "string",
                "description" : "The city that the customer wants to travel to",
            }
        },
    "required" : ["destination_city"],
    "additionalProperties" : False
    }
    }

set_price_function = {
    "name" : "set_ticket_price",
    "description" : "Set the price of a return ticket to the destination city.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city": {
                "type" : "string",
                "description" : "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The ticket price to set for the destination city"
            }
        },
    "required" : ["destination_city", "price"],
    "additionalProperties" : False
    }
    }

In [72]:
tools = [{"type" : "function", "function" : price_function}, {"type" : "function", "function" : set_price_function}]

In [73]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'set_ticket_price',
   'description': 'Set the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'},
     'price': {'type': 'number',
      'description': 'The ticket price to set for the destination city'}},
    'required': ['destination_city', 'price'],
    'additionalProperties': False}}}]

In [74]:
def handle_tool_calls(message):
    response = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            response.append({
                "role":"tool",
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
        elif tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price = arguments.get("price")
            set_ticket_price(city, price)
            response.append({
                "role":"tool",
                "content": "",
                "tool_call_id" : tool_call.id
            })
            
    return response


In [75]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [76]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7900
* To create a public link, set `share=True` in `launch()`.


In [33]:
import sqlite3

In [34]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices(city PRIMARY KEY, price REAL)')
    conn.commit()

In [59]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [77]:
get_ticket_price("Boston")

DATABASE TOOL CALLED: Getting price for Boston


'Ticket price to Boston is $300.0'

In [60]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [39]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [40]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [78]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7901
* To create a public link, set `share=True` in `launch()`.
